In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import polars as pl

In [2]:
#1.path
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"


print("PROJECT ROOT:", PROJECT_ROOT)
print("DATA ROOT:", DATA_ROOT)
print("EXTRACT ROOT:", EXTRACT_ROOT)

PROJECT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis
DATA ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data
EXTRACT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted


In [4]:
# 2. Find all Tournament parquet files


tournament_files = sorted(
    EXTRACT_ROOT.glob(
        "*/tournament_*.parquet"
    )
)


print("Number of Tournament files:",len(tournament_files))

Number of Tournament files: 35671


In [5]:
# 3. Inspect one Tournament parquet file

# Read the first tournament parquet file
test_tournament_df = pl.read_parquet(
    tournament_files[0]
)


# Display sample data

print(test_tournament_df)


# Display column names and data types

print("\nSchema:")
print(test_tournament_df.schema)

shape: (1, 16)
┌──────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ match_id ┆ tournament ┆ tournamen ┆ tournamen ┆ … ┆ has_perfo ┆ display_i ┆ priority ┆ competiti │
│ ---      ┆ _id        ┆ t_name    ┆ t_slug    ┆   ┆ rmance_gr ┆ nverse_ho ┆ ---      ┆ on_type   │
│ i64      ┆ ---        ┆ ---       ┆ ---       ┆   ┆ aph_featu ┆ me_away_t ┆ i64      ┆ ---       │
│          ┆ i64        ┆ str       ┆ str       ┆   ┆ re        ┆ eam…      ┆          ┆ i64       │
│          ┆            ┆           ┆           ┆   ┆ ---       ┆ ---       ┆          ┆           │
│          ┆            ┆           ┆           ┆   ┆ bool      ┆ bool      ┆          ┆           │
╞══════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ 11974053 ┆ 70826      ┆ Qualifier ┆ qualifier ┆ … ┆ false     ┆ false     ┆ 0        ┆ 2         │
│          ┆            ┆ s         ┆ s         ┆   ┆           ┆           

In [6]:
# 4. Check schemas of all Tournament files


schemas = Counter()


# Read every Tournament parquet file and collect schemas

for file in tournament_files:

    df = pl.read_parquet(file)

    schema_tuple = tuple(
        df.schema.items()
    )

    schemas[schema_tuple] += 1


print("Number of different schemas:",len(schemas))

Number of different schemas: 4


In [7]:
# 5.Display Tournament schema summary


for i, (schema, count) in enumerate(schemas.items(), start=1):

    print("=" * 60)

    print(f"Schema {i}")
    print(f"Number of files: {count}")

    for column, dtype in schema:
        print(f"{column} -> {dtype}")

Schema 1
Number of files: 77
match_id -> Int64
tournament_id -> Int64
tournament_name -> String
tournament_slug -> String
tournament_unique_id -> Null
tournament_category_name -> String
tournament_category_slug -> String
user_count -> Int64
ground_type -> Null
tennis_points -> Null
has_event_player_statistics -> Boolean
crowd_sourcing_enabled -> Boolean
has_performance_graph_feature -> Boolean
display_inverse_home_away_teams -> Boolean
priority -> Int64
competition_type -> Int64
Schema 2
Number of files: 4004
match_id -> Int64
tournament_id -> Int64
tournament_name -> String
tournament_slug -> String
tournament_unique_id -> Null
tournament_category_name -> String
tournament_category_slug -> String
user_count -> Int64
ground_type -> String
tennis_points -> Int64
has_event_player_statistics -> Boolean
crowd_sourcing_enabled -> Boolean
has_performance_graph_feature -> Boolean
display_inverse_home_away_teams -> Boolean
priority -> Int64
competition_type -> Int64
Schema 3
Number of files: 3

# Column Structure Comparison

This step compares only the column names across different schemas.
The goal is to verify whether all Tournament parquet files have the same structure.

If all schemas contain identical columns, schema differences are only caused by different data types (for example, Null vs Int64) and can be handled by applying a unified schema before concatenation

In [8]:
# 6.Display only column names for each schema


for i, (schema, count) in enumerate(schemas.items(), start=1):

    print(f"Schema {i} - Files: {count}")

    print(
        [column for column, dtype in schema]
    )

    print()

Schema 1 - Files: 77
['match_id', 'tournament_id', 'tournament_name', 'tournament_slug', 'tournament_unique_id', 'tournament_category_name', 'tournament_category_slug', 'user_count', 'ground_type', 'tennis_points', 'has_event_player_statistics', 'crowd_sourcing_enabled', 'has_performance_graph_feature', 'display_inverse_home_away_teams', 'priority', 'competition_type']

Schema 2 - Files: 4004
['match_id', 'tournament_id', 'tournament_name', 'tournament_slug', 'tournament_unique_id', 'tournament_category_name', 'tournament_category_slug', 'user_count', 'ground_type', 'tennis_points', 'has_event_player_statistics', 'crowd_sourcing_enabled', 'has_performance_graph_feature', 'display_inverse_home_away_teams', 'priority', 'competition_type']

Schema 3 - Files: 31105
['match_id', 'tournament_id', 'tournament_name', 'tournament_slug', 'tournament_unique_id', 'tournament_category_name', 'tournament_category_slug', 'user_count', 'ground_type', 'tennis_points', 'has_event_player_statistics', 'cr

In [9]:
# 7.Compare data types across Tournament schemas


# This step identifies columns with different data types
# across the four Tournament schemas.

column_dtypes = defaultdict(set)


# Collect all data types for each column

for schema, count in schemas.items():

    for column, dtype in schema:

        column_dtypes[column].add(str(dtype))


# Display columns that have different data types

for column, dtypes in column_dtypes.items():

    if len(dtypes) > 1:

        print(f"{column}: {dtypes}")

ground_type: {'Null', 'String'}
tennis_points: {'Null', 'Int64'}
competition_type: {'Null', 'Int64'}


In [10]:
# 8.Define standard schema for Tournament dataset


tournament_schema = {
    "match_id": pl.Int64,
    "tournament_id": pl.Int64,
    "tournament_name": pl.String,
    "tournament_slug": pl.String,
    "tournament_unique_id": pl.String,
    "tournament_category_name": pl.String,
    "tournament_category_slug": pl.String,
    "user_count": pl.Int64,
    "ground_type": pl.String,
    "tennis_points": pl.Int64,
    "has_event_player_statistics": pl.Boolean,
    "crowd_sourcing_enabled": pl.Boolean,
    "has_performance_graph_feature": pl.Boolean,
    "display_inverse_home_away_teams": pl.Boolean,
    "priority": pl.Int64,
    "competition_type": pl.Int64
}


print(tournament_schema)

{'match_id': Int64, 'tournament_id': Int64, 'tournament_name': String, 'tournament_slug': String, 'tournament_unique_id': String, 'tournament_category_name': String, 'tournament_category_slug': String, 'user_count': Int64, 'ground_type': String, 'tennis_points': Int64, 'has_event_player_statistics': Boolean, 'crowd_sourcing_enabled': Boolean, 'has_performance_graph_feature': Boolean, 'display_inverse_home_away_teams': Boolean, 'priority': Int64, 'competition_type': Int64}


In [11]:
# 9.Read and process all Tournament files


tournament_frames = []


# Read each Tournament parquet file

for file in tournament_files:

    df = pl.read_parquet(file)


    snapshot_date = file.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )

    # Apply the standard Tournament schema
    # strict=False keeps missing values as null

    df = df.cast(
        tournament_schema,
        strict=False
    )

    tournament_frames.append(df)


print("Number of processed files:",len(tournament_frames))

Number of processed files: 35671


In [12]:
# 10.Concatenate all Tournament dataframes


# Combine all processed Tournament dataframes vertically
# Each dataframe represents one Tournament snapshot

tournament_snapshot = pl.concat(
    tournament_frames,
    how="vertical"
)


print("Final shape:", tournament_snapshot.shape)
print("\nSchema:")
print(tournament_snapshot.schema)

Final shape: (35671, 17)

Schema:
Schema({'match_id': Int64, 'tournament_id': Int64, 'tournament_name': String, 'tournament_slug': String, 'tournament_unique_id': String, 'tournament_category_name': String, 'tournament_category_slug': String, 'user_count': Int64, 'ground_type': String, 'tennis_points': Int64, 'has_event_player_statistics': Boolean, 'crowd_sourcing_enabled': Boolean, 'has_performance_graph_feature': Boolean, 'display_inverse_home_away_teams': Boolean, 'priority': Int64, 'competition_type': Int64, 'snapshot_date': Date})


In [13]:
# 11.Check missing values in Tournament dataset


# Count the number of null values in each column

tournament_snapshot.null_count()

match_id,tournament_id,tournament_name,tournament_slug,tournament_unique_id,tournament_category_name,tournament_category_slug,user_count,ground_type,tennis_points,has_event_player_statistics,crowd_sourcing_enabled,has_performance_graph_feature,display_inverse_home_away_teams,priority,competition_type,snapshot_date
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,35671,0,0,0,562,31667,0,0,0,0,0,485,0


In [14]:
# 12.Check for duplicate rows


# Count rows that are completely duplicated

duplicate_count = (
    tournament_snapshot.height
    - tournament_snapshot.unique().height
)

print("Number of duplicate rows:",duplicate_count)

Number of duplicate rows: 0


In [15]:

# 13.Check match IDs with multiple snapshots


tournament_snapshot_counts = (
    tournament_snapshot
    .group_by("match_id")
    .agg(
        pl.col("snapshot_date")
        .n_unique()
        .alias("number_of_snapshots")
    )
)


multiple_tournament_snapshots = (
    tournament_snapshot_counts
    .filter(
        pl.col("number_of_snapshots") > 1
    )
)


print(
    "Match IDs with multiple snapshots:",
    multiple_tournament_snapshots.shape[0]
)


multiple_tournament_snapshots.head(10)

Match IDs with multiple snapshots: 16343


match_id,number_of_snapshots
i64,u32
12156255,2
12130707,2
12083520,2
12045211,2
12121266,2
12121236,2
12147594,2
12157571,2
12097939,2


In [18]:
# 14.Check Tournament information changes between snapshots


tournament_changes = (
    tournament_snapshot
    .group_by("match_id")
    .agg(
        pl.col("tournament_id")
        .n_unique()
        .alias("different_tournament_ids"),

        pl.col("tournament_name")
        .n_unique()
        .alias("different_names"),

        pl.col("tournament_category_name")
        .n_unique()
        .alias("different_categories"),

        pl.col("competition_type")
        .n_unique()
        .alias("different_competition_types")
    )
)


changed_tournament_matches = (
    tournament_changes
    .filter(
        (pl.col("different_tournament_ids") > 1)
        |
        (pl.col("different_names") > 1)
        |
        (pl.col("different_categories") > 1)
        |
        (pl.col("different_competition_types") > 1)
    )
)


print("Number of matches with Tournament changes:",changed_tournament_matches.shape[0])


changed_tournament_matches.head(10)

Number of matches with Tournament changes: 0


match_id,different_tournament_ids,different_names,different_categories,different_competition_types
i64,u32,u32,u32,u32


In [19]:
#15. Check Tournament ID consistency


tournament_id_consistency = (
    tournament_snapshot
    .group_by("tournament_id")
    .agg(
        pl.col("tournament_name")
        .n_unique()
        .alias("unique_names"),

        pl.col("tournament_category_name")
        .n_unique()
        .alias("unique_categories")
    )
)


inconsistent_tournaments = (
    tournament_id_consistency
    .filter(
        (pl.col("unique_names") > 1)
        |
        (pl.col("unique_categories") > 1)
    )
)


print("Tournament IDs with inconsistent information:",inconsistent_tournaments.shape[0])


inconsistent_tournaments.head(10)

Tournament IDs with inconsistent information: 0


tournament_id,unique_names,unique_categories
i64,u32,u32


In [20]:
# 16.Save cleaned Tournament dataset

clean_path = DATA_ROOT / "Data"

clean_path.mkdir(
    parents=True,
    exist_ok=True
)


tournament_snapshot.write_parquet(
    clean_path / "tournament_clean.parquet"
)


print("Saved successfully:")
print(clean_path / "tournament_clean.parquet")

Saved successfully:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/Data/tournament_clean.parquet


In [22]:
# 17.Verify saved Tournament dataset


tournament_clean = pl.read_parquet(
    clean_path / "tournament_clean.parquet"
)


print("Shape:")
print(tournament_clean.shape)


print("\nSchema:")
print(tournament_clean.schema)
tournament_clean.head(20)

Shape:
(35671, 17)

Schema:
Schema({'match_id': Int64, 'tournament_id': Int64, 'tournament_name': String, 'tournament_slug': String, 'tournament_unique_id': String, 'tournament_category_name': String, 'tournament_category_slug': String, 'user_count': Int64, 'ground_type': String, 'tennis_points': Int64, 'has_event_player_statistics': Boolean, 'crowd_sourcing_enabled': Boolean, 'has_performance_graph_feature': Boolean, 'display_inverse_home_away_teams': Boolean, 'priority': Int64, 'competition_type': Int64, 'snapshot_date': Date})


match_id,tournament_id,tournament_name,tournament_slug,tournament_unique_id,tournament_category_name,tournament_category_slug,user_count,ground_type,tennis_points,has_event_player_statistics,crowd_sourcing_enabled,has_performance_graph_feature,display_inverse_home_away_teams,priority,competition_type,snapshot_date
i64,i64,str,str,str,str,str,i64,str,i64,bool,bool,bool,bool,i64,i64,date
11974053,70826,"""Qualifiers""","""qualifiers""",null,"""Davis Cup""","""davis-cup""",6909,null,null,false,false,false,false,0,2,2024-02-01
11974066,70826,"""Qualifiers""","""qualifiers""",null,"""Davis Cup""","""davis-cup""",6909,null,null,false,false,false,false,0,2,2024-02-01
11998445,126168,"""Montpellier, France""","""montpellier-france""",null,"""ATP""","""atp""",2155,"""Hardcourt indoor""",250,false,false,false,false,0,2,2024-02-01
11998446,126168,"""Montpellier, France""","""montpellier-france""",null,"""ATP""","""atp""",2155,"""Hardcourt indoor""",250,false,false,false,false,0,2,2024-02-01
11998447,126168,"""Montpellier, France""","""montpellier-france""",null,"""ATP""","""atp""",2155,"""Hardcourt indoor""",250,false,false,false,false,0,2,2024-02-01
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
11998672,126174,"""Burnie 1, Australia""","""burnie-1-australia""",null,"""Challenger""","""challenger""",125,"""Hardcourt outdoor""",null,false,false,false,false,0,2,2024-02-01
11998674,126174,"""Burnie 1, Australia""","""burnie-1-australia""",null,"""Challenger""","""challenger""",125,"""Hardcourt outdoor""",null,false,false,false,false,0,2,2024-02-01
11998675,126174,"""Burnie 1, Australia""","""burnie-1-australia""",null,"""Challenger""","""challenger""",125,"""Hardcourt outdoor""",null,false,false,false,false,0,2,2024-02-01


In [24]:
# 18.Verify saved Venue dataset


venue_clean = pl.read_parquet(
    clean_path / "venue_clean.parquet"
)


print("Shape:")
print(venue_clean.shape)


print("\nSchema:")
print(venue_clean.schema)
venue_clean.head(20)

Shape:
(35423, 6)

Schema:
Schema({'match_id': Int64, 'city': String, 'stadium': String, 'venue_id': Int64, 'country': String, 'snapshot_date': Date})


match_id,city,stadium,venue_id,country,snapshot_date
i64,str,str,i64,str,date
11974053,"""Groningen""","""Martini Plaza""",7324,"""Netherlands""",2024-02-01
11974066,"""Vilnius""","""SEB Arena""",36345,"""Lithuania""",2024-02-01
11998445,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
11998446,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
11998447,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
…,…,…,…,…,…
11998672,"""Burnie""","""Ct 5""",20229,"""Australia""",2024-02-01
11998674,"""Burnie""","""Centre Court""",24474,"""Australia""",2024-02-01
11998675,"""Burnie""","""Centre Court""",24474,"""Australia""",2024-02-01
